In [1]:
import os
import sys
import json
from tqdm import tqdm
import shutil
import numpy as np
import pandas as pd
import cv2 as cv
import csv
from tqdm import tqdm
sys.path.insert(0, "../../packages/python")
from models import cell_segmentation as segmentators

2025-07-27 09:44:48.452927: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-27 09:44:48.458908: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753620288.465827   29787 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753620288.468001   29787 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753620288.473578   29787 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
IMG_TARGET_SIDE = 300

CROPS_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing/'

IMAGES_PATH_1 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/images/'
IMAGES_PATH_2 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/onion_cell_merged/images/train/'
IMAGES_PATH_3 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/onion_cell_merged/images/test/'
IMAGES_PATH_4 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/onion_cell_merged/images/valid/'

CSV_PATH_1 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/ina/data/'
CSV_PATH_2 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/data/train/'
CSV_PATH_3 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/data/test/'
CSV_PATH_4 = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/onion_cell_merged/data/valid/'

JSON_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/temp/datasets_area_data.json'

OUTPUT_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_enlarged'
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [3]:
crops = [CROPS_PATH + crops_path for crops_path in sorted(os.listdir(CROPS_PATH))]

csvs_1 = [CSV_PATH_1 + csvs_path for csvs_path in sorted(os.listdir(CSV_PATH_1))]
csvs_2 = [CSV_PATH_2 + csvs_path for csvs_path in sorted(os.listdir(CSV_PATH_2))]
csvs_3 = [CSV_PATH_3 + csvs_path for csvs_path in sorted(os.listdir(CSV_PATH_3))]
csvs_4 = [CSV_PATH_4 + csvs_path for csvs_path in sorted(os.listdir(CSV_PATH_4))]
csvs = csvs_1 + csvs_2 + csvs_3 + csvs_4

images_1 = [IMAGES_PATH_1 + images_path for images_path in sorted(os.listdir(IMAGES_PATH_1))]
images_2 = [IMAGES_PATH_2 + images_path for images_path in sorted(os.listdir(IMAGES_PATH_2))]
images_3 = [IMAGES_PATH_3 + images_path for images_path in sorted(os.listdir(IMAGES_PATH_3))]
images_4 = [IMAGES_PATH_4 + images_path for images_path in sorted(os.listdir(IMAGES_PATH_4))]
images = images_1 + images_2 + images_3 + images_4

with open(JSON_PATH, 'r') as f: #json with the information of the filename of the images
    area_data = json.load(f)

In [5]:
for crop_path in tqdm(crops):
    # 1. Extract IDs from crop name
    crop_filename = os.path.basename(crop_path)
    subsection, image_name, cell_id = crop_filename.split('_')
    cell_id = cell_id.split('.')[0]  # Remove extension if present

    # 2. Find the corresponding image
    image_filename = f"{subsection}_{image_name}"
    image_path = None
    for img_path in images:
        if image_filename in img_path:
            image_path = img_path
            break

    if not image_path:
        print(f"Warning: Image not found for crop {crop_filename}")
        continue

    # 3. Find the corresponding CSV
    csv_filename = f"{subsection}_{image_name}.csv"
    csv_path = None
    for csv_file in csvs:
        if csv_filename in csv_file:
            csv_path = csv_file
            break

    if not csv_path:
        print(f"Warning: CSV not found for crop {crop_filename}")
        continue

    # 4. Read the CSV and extract cell data
    try:
        df = pd.read_csv(csv_path)
        df_bbox = df[df['cell_id'] == int(cell_id)]  # Pandas reads cell_id as int
        if not df_bbox.empty:

            resize_factor = IMG_TARGET_SIDE/area_data['INA']['lado_cuadrado']

            image_group = subsection[0] if subsection[0].isalpha() else "INA" 
            image_side = area_data[image_group]['lado_cuadrado']
            image_resize_factor = int(resize_factor * image_side)

            img = cv.imread(image_path)
            df = pd.read_csv(csv_path)

            for _, row in df_bbox.iterrows():
                x, y, w, h = row['x'], row['y'], row['w'], row['h']
                x, y, w, h = segmentators.CellMaskGenerator.adjust_bbox(segmentators.CellMaskGenerator, x, y, w, h, image_resize_factor*image_resize_factor, img.shape[1], img.shape[0])

                crop = cv.resize(img[y:y+h, x:x+w], (IMG_TARGET_SIDE, IMG_TARGET_SIDE))
                output_path = os.path.join(OUTPUT_PATH, crop_filename)
                cv.imwrite(output_path, crop)
        else:
            print(f"Warning: Cell ID {cell_id} not found in CSV {csv_filename}")
    except Exception as e:
        print(f"An error occurred while reading CSV: {e}")


100%|██████████| 8697/8697 [01:45<00:00, 82.52it/s] 


In [34]:
import os
import shutil

def update_images_from_source(source_folder: str, destination_folder: str):
    """
    Updates images in a destination folder from a source folder.

    This function iterates through the files in the destination_folder (your subset)
    and for each file, it copies the corresponding file from the source_folder
    (your full set), overwriting the old one.

    Args:
        source_folder (str): The path to the folder containing the full
                             set of updated images (Folder A).
        destination_folder (str): The path to the folder containing the subset
                                  of images to be updated (Folder B).
    """
    print(f"Starting update of images in '{destination_folder}' from '{source_folder}'...")

    # Get the list of image filenames in the destination folder (Folder B)
    try:
        images_to_update = os.listdir(destination_folder)
    except FileNotFoundError:
        print(f"Error: Destination folder not found at '{destination_folder}'")
        return

    if not images_to_update:
        print(f"No images found in '{destination_folder}'. Nothing to do.")
        return

    updated_count = 0
    not_found_count = 0

    # Iterate through each image name found in Folder B
    for image_name in images_to_update:
        source_image_path = os.path.join(source_folder, image_name)
        destination_image_path = os.path.join(destination_folder, image_name)

        # Check if the corresponding image exists in the source folder (Folder A)
        if os.path.exists(source_image_path):
            try:
                # Copy the updated image from A to B, overwriting the old one.
                # shutil.copy2 also copies file metadata.
                shutil.copy2(source_image_path, destination_image_path)
                print(f"  - Updated: {image_name}")
                updated_count += 1
            except Exception as e:
                print(f"  - Error copying {image_name}: {e}")
        else:
            print(f"  - Warning: '{image_name}' not found in source folder. Skipping.")
            not_found_count += 1

    print("\nUpdate complete.")
    print(f"Successfully updated {updated_count} images.")
    if not_found_count > 0:
        print(f"{not_found_count} images from Folder B were not found in Folder A.")

# --- How to use this script ---

# 1. Replace these placeholder paths with the actual paths to your folders.
#    On Windows, your paths might look like: 'C:\Users\YourUser\Desktop\FolderA'
#    On macOS or Linux: '/home/user/Documents/FolderA'

FOLDER_A_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_enlarged'
FOLDER_B_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_rf_enlarged/'

# 2. Run the function.
update_images_from_source(FOLDER_A_PATH, FOLDER_B_PATH)


Starting update of images in '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_rf_enlarged/' from '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_enlarged'...
  - Updated: A_40_90.png
  - Updated: D_55_7.png
  - Updated: B_42_41.png
  - Updated: B_46_67.png
  - Updated: A_326_67.png
  - Updated: B_3_29.png
  - Updated: B_35_65.png
  - Updated: B_38_2.png
  - Updated: Entrega1_00009_182.png
  - Updated: A_4_63.png
  - Updated: A_102_74.png
  - Updated: A_98_70.png
  - Updated: A_70_102.png
  - Updated: B_118_8.png
  - Updated: A_117_114.png
  - Updated: D_30_8.png
  - Updated: A_5_64.png
  - Updated: A_306_35.png
  - Updated: B_70_43.png
  - Updated: A_52_54.png
  - Updated: B_46_10.png
  - Updated: A_317_37.png
  - Updated: A_241_19.png
  - Updated: A_59_17.png
  - Updated: B_16_10.png
  - Updated: A_27_43.png
  - Updated: F_81_87.png
  - Updated: A_23_42.png
  - Updated: A_41_47.png
  - Updated: F_79_56.png
  - Updated: A_40_30.p

In [6]:
import os
import random
import shutil

def copy_random_images(source_folder, destination_folder, num_images=100):
    """
    Copies a specified number of random image files from a source folder to a destination folder.

    Args:
        source_folder (str): The path to the folder containing the images.
        destination_folder (str): The path to the folder where images will be copied.
        num_images (int): The number of random images to copy.
    """
    if not os.path.exists(source_folder):
        print(f"Source folder '{source_folder}' does not exist.")
        return

    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)
        print(f"Created destination folder: '{destination_folder}'")

    image_files = [f for f in os.listdir(source_folder) if os.path.isfile(os.path.join(source_folder, f)) and f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'))]

    if not image_files:
        print(f"No image files found in '{source_folder}'.")
        return

    if len(image_files) < num_images:
        print(f"Warning: Only {len(image_files)} image(s) available, copying all of them instead of {num_images}.")
        num_images = len(image_files)

    selected_images = random.sample(image_files, num_images)

    print(f"Copying {len(selected_images)} random images from '{source_folder}' to '{destination_folder}'...")
    for image_name in selected_images:
        source_path = os.path.join(source_folder, image_name)
        destination_path = os.path.join(destination_folder, image_name)
        try:
            shutil.copy2(source_path, destination_path)
            print(f"Copied: {image_name}")
        except Exception as e:
            print(f"Error copying {image_name}: {e}")

    print("Image copying complete.")

FOLDER_A_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_enlarged'
FOLDER_B_PATH = '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/upload_zooniverse/'

copy_random_images(FOLDER_A_PATH, FOLDER_B_PATH)

Copying 100 random images from '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_enlarged' to '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/upload_zooniverse/'...
Copied: F_24_4.png
Copied: 003_00061_1.png
Copied: C_6_5.png
Copied: 003_00060_37.png
Copied: 002_00063_4.png
Copied: A_349_4.png
Copied: 003_00076_13.png
Copied: 004_00004_27.png
Copied: A_71_21.png
Copied: B_118_8.png
Copied: A_298_75.png
Copied: 003_00021_94.png
Copied: A_265_10.png
Copied: 003_00004_20.png
Copied: 001_00082_85.png
Copied: 002_00007_17.png
Copied: B_34_32.png
Copied: 003_00044_41.png
Copied: 002_00062_33.png
Copied: D_8_16.png
Copied: 004_00001_3.png
Copied: F_21_61.png
Copied: 003_00025_40.png
Copied: B_117_8.png
Copied: B_109_15.png
Copied: 004_00026_4.png
Copied: 004_00075_207.png
Copied: 001_00078_31.png
Copied: 001_00035_64.png
Copied: 003_00080_105.png
Copied: A_105_11.png
Copied: F_86_4.png
Copied: 004_00021_176.png
Copied: 004_00039_5.png
Copied: 001